# 08 · Accelerate — M×N GPU (subprocess.Popen)

`accelerate launch` CLI 를 `subprocess.Popen` 으로 driver 에서 띄운다. 노트북 매직(`!`, `%sh`)은 PATH/auth 문제로 노트북-스코프 env 와 충돌하기 쉬워, 명령 문자열을 `shlex.split` → `Popen` 으로 직접 실행하고 자식 프로세스에 `DATABRICKS_HOST`/`DATABRICKS_TOKEN` 을 명시 주입한다.

`--num_processes` 는 지정하지 않는다. Accelerate 가 가시 GPU 수만큼 자동으로 띄운다. 토폴로지 1×1 / 1×N / M×N 은 클러스터 사양으로만 결정되며, 이 노트북 한 개로 전부 커버한다.

Entry point 는 [`torch_distributor_trainer.py`](https://github.com/Aiden-Jeon/databricks-distributed-training/blob/main/02-script-based/torch_distributor_trainer.py) — TorchDistributor / Accelerate 가 공통으로 사용하도록 설계됨 (`if __name__ == '__main__':` + argparse).

**클러스터**: Classic GPU, DBR 17.3 LTS ML. 단일 fat-node (예: `g5.12xlarge` 4×A10) 또는 multi-node 환경에서 그대로 동작.

In [ ]:
%run ./00-setup

## Entry point 스크립트 경로

노트북과 같은 워크스페이스 폴더의 `torch_distributor_trainer.py` 를 `accelerate launch` 대상으로 넘긴다.

In [ ]:
import os

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
SCRIPT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
SCRIPT_PATH = os.path.join(SCRIPT_DIR, "torch_distributor_trainer.py")
print(f"SCRIPT_PATH={SCRIPT_PATH}")

## MLflow run 생성

driver 에서 run 을 시작하고 `run_id` 만 빼서 자식 프로세스에 넘긴다. rank 0 자식이 같은 run 에 attach 해서 system metrics / per-epoch 메트릭을 누적한다.

In [ ]:
import mlflow

cfg = CONFIG
mlflow.set_experiment(EXPERIMENT_PATH)
with mlflow.start_run(run_name="acc-MxN", log_system_metrics=True) as run:
    RUN_ID = run.info.run_id
print(f"RUN_ID={RUN_ID}")

## `accelerate launch` 명령 + subprocess.Popen

1. `inference_cmd` 를 f-string 으로 조립하고 `shlex.split` 으로 토큰화.
2. `sub_env` 에 `DATABRICKS_HOST` / `DATABRICKS_TOKEN` 주입 — 자식 프로세스는 노트북 auth context 를 상속하지 않으므로, MLflow / WorkspaceClient 호출을 위해 명시 전달.
3. Py4J keepalive 스레드 — 장시간 학습 중 driver Py4J gateway 가 끊겨 후처리가 폭사하는 것을 막기 위해 5분마다 `SELECT 1` 을 찌른다.
4. `stderr=STDOUT` + `bufsize=1` + `text=True` 로 stdout/stderr 한 스트림 라인 단위 출력.
5. non-zero return 시 `dbutils.notebook.exit` 로 실패 종료 → DAB job retry 가 동작.

In [ ]:
import shlex
import subprocess
import threading

tower_hidden_args = " ".join(str(x) for x in cfg["tower_hidden"])
ckpt_path = os.path.join(CKPT_DIR, "acc-MxN.pt")

inference_cmd = f"""accelerate launch --mixed_precision=bf16 {SCRIPT_PATH} \
    --run_id '{RUN_ID}' \
    --db_host '{DB_HOST}' \
    --db_token '{DB_TOKEN}' \
    --data_dir '{DATA_DIR}' \
    --ckpt_path '{ckpt_path}' \
    --n_users {cfg['n_users']} \
    --n_items {cfg['n_items']} \
    --emb_dim {cfg['emb_dim']} \
    --tower_hidden {tower_hidden_args} \
    --batch_size {cfg['batch_size_per_gpu']} \
    --num_epochs {cfg['num_epochs']} \
    --max_steps_per_epoch {cfg['max_steps_per_epoch']} \
    --patience {PATIENCE} \
    --min_delta {MIN_DELTA} \
    --topology MxN \
    --script_dir '{SCRIPT_DIR}'
"""

sub_env = os.environ.copy()
sub_env["DATABRICKS_HOST"] = DB_HOST
sub_env["DATABRICKS_TOKEN"] = DB_TOKEN

done_event = threading.Event()

def _keepalive(interval=300):
    while not done_event.is_set():
        try:
            spark.sql("SELECT 1").collect()
        except Exception:
            pass
        done_event.wait(interval)

keepalive_thread = threading.Thread(target=_keepalive, daemon=True)
keepalive_thread.start()

p = subprocess.Popen(
    shlex.split(inference_cmd),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=sub_env,
)

for line in p.stdout:
    print(line, end="")

p.wait()
done_event.set()

if p.returncode != 0:
    dbutils.notebook.exit(f"accelerate launch failed: rc={p.returncode}")